# AI Person Detection with YOLOv8 (Google Colab – GPU T4)

Notebook นี้ออกแบบสำหรับการสาธิต:
- ตรวจจับ **บุคคล (Person Detection)** จากภาพ/วิดีโอ
- ใช้ **YOLOv8n** (เร็ว เหมาะกับ Colab T4)
- แปลงวิดีโอ Output เป็น **MP4** เพื่อดูในเว็บได้
- สาธิต **Fine-tune** ด้วย Public Dataset (COCO8)

ก่อนเริ่ม: ไปที่ **Runtime → Change runtime type → GPU** แล้วกด Save


In [ ]:
# 1) Install dependencies
!pip -q install ultralytics opencv-python ffmpeg-python
import ultralytics
ultralytics.checks()

In [ ]:
# 2) Check GPU (ต้องเห็น Tesla T4)
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3) Load YOLOv8 model
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model

## 4) Person Detection on an Image
อัปโหลดภาพของคุณเอง (แนะนำ: ไม่ระบุตัวตน / เบลอหน้า)

In [ ]:
from google.colab import files
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
img_path

In [ ]:
# Detect only 'person' (COCO class 0)
results = model.predict(source=img_path, classes=[0], conf=0.25, device=0, save=True)

import matplotlib.pyplot as plt
img = results[0].plot()
plt.figure(figsize=(10,6))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5) Person Detection on a Video (Convert to MP4)
วิดีโอ Output จะถูกแปลงเป็น MP4 เพื่อดูใน Google Colab ได้

In [ ]:
from google.colab import files
uploaded = files.upload()
vid_path = next(iter(uploaded.keys()))
vid_path

In [ ]:
results = model.predict(source=vid_path, classes=[0], conf=0.25, device=0, save=True)

import glob, os
cands = sorted(glob.glob('runs/detect/predict*/**/*.*', recursive=True))

video_out = None
for p in reversed(cands):
    if os.path.splitext(p)[1].lower() in ['.avi', '.mp4', '.mov', '.mkv']:
        video_out = p
        break

print('Raw output file:', video_out)

In [ ]:
# Convert to MP4 for web preview
mp4_out = 'output.mp4'
!ffmpeg -y -i "{video_out}" -vcodec libx264 -acodec aac {mp4_out}
mp4_out

In [ ]:
from IPython.display import Video, display
display(Video(mp4_out, embed=True))

In [ ]:
from google.colab import files
files.download(mp4_out)

## 6) Fine-tune YOLOv8 with Public Dataset (COCO8)
ใช้ COCO8 ซึ่งเป็น public dataset ขนาดเล็ก เหมาะกับการสาธิต

In [ ]:
from ultralytics.utils.downloads import download
download('https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip')
!unzip -q coco8.zip
!ls

In [ ]:
from ultralytics import YOLO
ft_model = YOLO('yolov8n.pt')
ft_model.train(data='coco8.yaml', epochs=3, imgsz=640, batch=16, device=0)